In [1]:
# import dependencies
import RMF
import IMP.npctransport
import pickle
import numpy as np
import os

## NTRs
Reads data from NPC simulations and saves trajectories for each NTR.

In general, pickles generated by this script contain numpy arrays with the shape:

[kap_amount, 3, n_frames]

In [2]:
def _has_depth_with_site(root, i):
    """ returns true if node subtree thru first child is at least i
        levels, including the root node itself, and the lead is a site """
#  print root, i, len(root.get_children())
    if (i==1) and root.get_name()=="site":
        return True
    c = root.get_children()
    if len(c) == 0:
        return False
    return _has_depth_with_site(c[0], i-1)

def _add_nodes(node, tf, type_prefixes, depth=0):
    '''
    node - rmf node to scan
    tf - typed factory
    type_prefixes - list of full type prefixes (e.g. "Nup1" for "Nup1N")

    adds only nodes whose type name begins with any of the specified type prefixes
    '''
    children = node.get_children()
    ret = []
    if len(children)==0:
        return ret
    if _has_depth_with_site(node, 3) and tf.get_is(children[0]):
        child_type = tf.get(children[0]).get_type_name()
        if any([child_type.startswith(tp) for tp in type_prefixes]):
            ret.append(children)
    for c in children:
        ret += _add_nodes(c, tf,  type_prefixes, depth+1)
    return ret

def _add_nodes_exact(node, tf, types, depth=0):
    '''
    node - rmf node to scan
    tf - typed factory
    types - list of full types (e.g. "Nup1N")

    adds only nodes whose type name begins with any of the specified type prefixes
    '''
    children = node.get_children()
    ret = []
    if len(children)==0:
        return ret
    if _has_depth_with_site(node, 3) and tf.get_is(children[0]):
        child_type = tf.get(children[0]).get_type_name()
        if any([child_type == tp for tp in types]):
            ret.append(children)
    for c in children:
        ret += _add_nodes_exact(c, tf,  types, depth+1)
    return ret

def get_trajectories_shape(n_molecules, start_t, end_t, step_t, frames_per_file):
    return [n_molecules, 3, int(((end_t - start_t) / step_t) * frames_per_file)]

def load_kap_data(input_rmf_path, kap_radius, kap_amount, start_t, end_t, step_t, frames_per_file=104, one_frame_from_each=False):
    trajectories_shape = [kap_amount, 3, int(((end_t - start_t) / step_t) * frames_per_file)]
    trajectories = np.zeros(shape=trajectories_shape)

    for rmf_t in range(start_t, end_t, step_t):
        in_fh = RMF.open_rmf_file_read_only(f"{input_rmf_path}/{rmf_t}.movie.rmf")
        rff = RMF.ReferenceFrameFactory(in_fh)
        tf = RMF.TypedFactory(in_fh)
        kap_types = [f"kap{kap_radius}"]

        # load data
        type2chains={}
        for i, kap_type in enumerate(kap_types):
            type2chains[kap_type] = _add_nodes(in_fh.get_root_node(), tf, [kap_type])
            
        # set frame
        for f_id, f in enumerate(in_fh.get_frames()):
            in_fh.set_current_frame(f)
    
            traj_i = int(f_id + ((rmf_t - start_t) / step_t) * frames_per_file)
            # read data
            for kap_i in range(kap_amount):
                coord = rff.get(type2chains["kap35"][0][kap_i]).get_translation()
                
                trajectories[kap_i, 0, traj_i] = coord[0] / 10
                trajectories[kap_i, 1, traj_i] = coord[1] / 10
                trajectories[kap_i, 2, traj_i] = coord[2] / 10
            if one_frame_from_each:
                break
    return trajectories

def _get_fg_types(input_pb_path):
    fg_types = []
    output = IMP.npctransport.Output()
    FILE = open(input_pb_path,"rb")
    output.ParseFromString(FILE.read())
    a = output.assignment
    for fg in a.fgs:
        fg_types.append(fg.type)
    return fg_types

def load_fg_data(input_rmf_path, fg_types, n_chains_per_fg, n_beads_per_fg, start_t, end_t, step_t, frames_per_file=104, one_frame_from_each=False):
    trajectories = dict()
    for i, fg_type in enumerate(fg_types):
        shape = [n_chains_per_fg[i], n_beads_per_fg[i], 3, int(((end_t - start_t) / step_t) * frames_per_file)]
        trajectories[fg_type] = np.zeros(shape=shape)

    for rmf_t in range(start_t, end_t, step_t):
        in_fh = RMF.open_rmf_file_read_only(f"{input_rmf_path}/{rmf_t}.movie.rmf")
        rff = RMF.ReferenceFrameFactory(in_fh)
        tf = RMF.TypedFactory(in_fh)
        type2chains={}
        for i, fg in enumerate(fg_types):
            type2chains[fg] = _add_nodes(in_fh.get_root_node(), tf, [fg + "anchor", fg + "s", fg + "C", fg + "N", fg + "m"])
            
        for f_id, f in enumerate(in_fh.get_frames()):
            # set frame
            in_fh.set_current_frame(f)
    
            traj_i = int(f_id + ((rmf_t - start_t) / step_t) * frames_per_file)
            # read data
            for nup_i, fg_type in enumerate(fg_types):
                for chain_i in range(n_chains_per_fg[nup_i]):
                    for bead_i in range(n_beads_per_fg[nup_i]):
                        try:
                            coord = rff.get(type2chains[fg_type][chain_i][bead_i]).get_translation()
                            trajectories[fg_type][chain_i, bead_i, 0, traj_i] = coord[0] / 10
                            trajectories[fg_type][chain_i, bead_i, 1, traj_i] = coord[1] / 10
                            trajectories[fg_type][chain_i, bead_i, 2, traj_i] = coord[2] / 10
                        except:
                            pass
            if one_frame_from_each:
                break
    return trajectories    

In [27]:
trajectories = load_kap_data(
          input_rmf_path="/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/2/",
          kap_radius=35,
          kap_amount=250,
          start_t=40000,
          end_t=80000,
          step_t=100,
          frames_per_file=1,
          one_frame_from_each=True)
with open("spatial_markov_chain_data.pickle", "wb") as f:
    pickle.dump(trajectories, f)

In [6]:
n_molecules = 250
start_t = 160000
end_t = 170000
step_t = 100
frames_per_file = 1
sims_range = range(1, 51)

bad_sims = []
for i in sims_range:
    # check that trajectory reached end_t time
    if not os.path.isfile(f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/divergences/{i}/{end_t}.movie.rmf"):
        print(f"sim {i} not long enough")
        bad_sims.append(i)
        continue

# good_sims = [i for i in sims_range if i not in bad_sims]
# # diverged_trajs = np.zeros(shape=[len(good_sims)] + get_trajectories_shape(kap_amount, start_t, end_t, step_t, frames_per_file))
# for i in good_sims:
#     print(f"loading sim {i}")
#     trajectories = load_data(
#           input_rmf_path=f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/divergences/{i}",
#           kap_radius=35,
#           kap_amount=kap_amount,
#           start_t=start_t,
#           end_t=end_t,
#           step_t=step_t,
#           frames_per_file=frames_per_file,
#           one_frame_from_each=True)
#     with open(f"data/singles/{i}/150-160.pickle", "wb") as f:
#         pickle.dump(trajectories, f)

sim 13 not long enough
sim 14 not long enough
sim 15 not long enough
sim 16 not long enough


In [10]:
# Merge singles into merged

arrays = []
for i in range(1, 51):
    with open(f"data/singles/{i}/130-140.pickle", "rb") as f:
        arrays.append(pickle.load(f))
trajectories = np.concatenate(arrays, axis=0)
with open(f"data/merged/130-140.pickle", "wb") as f:
    pickle.dump(trajectories, f)

In [11]:
# Merge merged

arrays = []
for i in ["110-130", "130-140", "140-150", "150-160"]:
    with open(f"data/merged/{i}.pickle", "rb") as f:
        arrays.append(pickle.load(f))
trajectories = np.concatenate(arrays, axis=2)
with open(f"data/merged/110-160.pickle", "wb") as f:
    pickle.dump(trajectories, f)

In [7]:
# Merge times

for j in range(1, 51):
    arrays = []
    for i in ["110-130", "130-140", "140-150", "150-160"]:
        with open(f"data/singles/{j}/{i}.pickle", "rb") as f:
            arrays.append(pickle.load(f))
    trajectories = np.concatenate(arrays, axis=2)
    with open(f"data/singles/{j}/110-160.pickle", "wb") as f:
        pickle.dump(trajectories, f)

## FGs

Reads data from NPC simulations and saves trajectories for each NUP.

In general, pickles generated by this script contain the following structurs:

Dictionary(nup_type : [chains, beads, 3, n_frames])

```
Nsp1: 48 chains, each has 32 beads
Nup100: 16 chains, each has 40 beads
Nup116: 16 chains, each has 48 beads
Nup159: 16 chains, each has 34 beads
Nup49: 32 chains, each has 14 beads
Nup57: 32 chains, each has 15 beads
Nup145: 16 chains, each has 13 beads
Nup1: 72 chains, each has 40 beads
Nup60: 16 chains, each has 12 beads

Total: 264 Chains, 7696 Beads
```

In [ ]:
fg_types = ['Nsp1', 'Nup100', 'Nup116', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60']
n_chains_per_fg = [48, 16, 16, 16, 32, 32, 16, 72, 16]
n_beads_per_fg = [32, 40, 48, 34, 14, 15, 13, 40, 12]
start_t = 110000
end_t = 130000
step_t = 100
frames_per_file = 1
sims_range = range(1, 51)

bad_sims = []
for i in sims_range:
    # check that trajectory reached end_t time
    if not os.path.isfile(f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/divergences/{i}/{end_t}.movie.rmf"):
        print(f"sim {i} not long enough")
        bad_sims.append(i)
        continue

good_sims = [i for i in sims_range if i not in bad_sims]
# diverged_trajs = np.zeros(shape=[len(good_sims)] + get_trajectories_shape(kap_amount, start_t, end_t, step_t, frames_per_file))
for i in good_sims:
    print(f"loading sim {i}")
    trajectories = load_fg_data(input_rmf_path=f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/divergences/{1}/",
            fg_types=fg_types,
            n_chains_per_fg=n_chains_per_fg,
            n_beads_per_fg=n_beads_per_fg,
            start_t=start_t,
            end_t=end_t,
            step_t=100,
            frames_per_file=1,
            one_frame_from_each=True)
    with open(f"data/singles/{i}/110-130-fgs.pickle", "wb") as f:
        pickle.dump(trajectories, f)



In [6]:
# Merge times (most useful)


for j in range(24, 51):
    print(j)
    trajectories = {'Nsp1':[], 'Nup100':[], 'Nup116':[], 'Nup159':[], 'Nup49':[], 'Nup57':[], 'Nup145':[], 'Nup1':[], 'Nup60':[]}
    for i in ["110-130", "130-140", "140-150", "150-160"]:
        with open(f"data/singles/{j}/{i}-fgs.pickle", "rb") as f:
            temp = pickle.load(f)
            for key, val in temp.items():
                trajectories[key].append(val)
    for key, val in trajectories.items():
        trajectories[key] = np.concatenate(val, axis=3)
    with open(f"data/singles/{j}/110-160-fgs.pickle", "wb") as f:
        pickle.dump(trajectories, f)

24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50


In [3]:
# Merge singles into merged

trajectories = {'Nsp1':[], 'Nup100':[], 'Nup116':[], 'Nup159':[], 'Nup49':[], 'Nup57':[], 'Nup145':[], 'Nup1':[], 'Nup60':[]}
for i in range(1, 51):
    with open(f"data/singles/{i}/110-130-fgs.pickle", "rb") as f:
        temp = pickle.load(f)
        for key, val in temp.items():
            trajectories[key].append(val)
for key, val in trajectories.items():
    trajectories[key] = np.concatenate(val, axis=0)
with open(f"data/merged/110-130-fgs.pickle", "wb") as f:
    pickle.dump(trajectories, f)

In [4]:
# Merge merged

trajectories = {'Nsp1':[], 'Nup100':[], 'Nup116':[], 'Nup159':[], 'Nup49':[], 'Nup57':[], 'Nup145':[], 'Nup1':[], 'Nup60':[]}
for i in ["110-130", "130-140", "140-150", "150-160"]:
    with open(f"data/merged/{i}-fgs.pickle", "rb") as f:
        temp = pickle.load(f)
        for key, val in temp.items():
            trajectories[key].append(val)
for key, val in trajectories.items():
    trajectories[key] = np.concatenate(val, axis=3)
with open(f"data/merged/110-160-fgs.pickle", "wb") as f:
    pickle.dump(trajectories, f)

: 